# Kenya maize planting-window pipeline — end to end

Runs the whole workflow in JupyterLab:

1. **GEE pipeline** — S2 red-edge + S1 SAR + FPAR fusion → SOS → planting dekad (+ WRSI), exported to Drive
2. **Local maps** — render the exported GeoTIFFs into a styled PDF/PNG (no GEE, no quota)
3. **Admin skill** — county / constituency / ward stats + skill vs the FEWS/FAO calendar window
4. **AEZ maturity** — overlay planting dekad with Kenya AEZ → early / medium / late-maturing maize

> Stages 2–4 run **locally** from the GeoTIFF tiles and are unaffected by the Earth Engine quota.
> Only Stage 1 touches GEE (and is throttle-gated on the free tier).

> **Fusion & scale (revised methodology):** the production greenness is **cue fusion** (NDRE + FPAR primary, SAR gap-fill). **ubESTARFM was tested and shelved** — a documented negative result for onset (−3 to −4 pts vs cue at matched 250 m; see `UBESTARFM_FINDING.md`). **Resolution is the dominant skill lever** (10 m→250 m ≈ −11 pts), and two concurrent 10 m country exports do not sustain on GEE, so **250 m is the production scale**.

## 0. Configuration (edit these paths if you move things)

### Stage 0 · Configuration, and what is different about this notebook

**This is the local workflow, not a Colab notebook.** Paths are absolute to a laptop. Only stage 1
touches Earth Engine; stages 2 to 5 run from the exported GeoTIFFs with rasterio and matplotlib, so
they are unaffected by Earth Engine quota and can be re-run freely.

| Setting | Meaning |
|---|---|
| `PRODUCT` | the export name the later stages look for |
| `CAL_WIN` | the FEWS and FAO indicative planting window, in dekads, that skill is scored against |
| `AEZ_SHP` | Kenya's Jaetzold and Sombroek agro-ecological zones |

**`CAL_WIN = (8, 12)`** is dekad 8 to 12, that is 11 March to 30 April. Skill is the share of maize
area whose estimated planting dekad falls inside it. Change it with the season: short rains 28 to 32,
Ethiopia Meher 10 to 15.

In [ ]:
import os

PROJECT      = 'ee-manzikye'                                                   # your GEE cloud project
PIPELINE_DIR = '/Users/hildamanzi/Downloads/planting_pipeline'                 # where the .py scripts live
OUTPUTS_DIR  = '/Users/hildamanzi/ICPAC-WORK/plantingwindow_pipeline/planting_outputs'  # exported GeoTIFF tiles
AEZ_SHP      = '/Users/hildamanzi/AEZ-COUNTRIES/KENYA_AEZ/kenya_aezones.shp'   # Kenya AEZ polygons

YEAR, COUNTRY, CROP = 2024, 'Kenya', 'maize'
PRODUCT   = f'planting_{COUNTRY}_{CROP}_Longrains_{YEAR}'
CAL_WIN   = (8, 12)     # FEWS/FAO indicative planting window (dekads); Short rains = (28, 32)

os.environ['EE_PROJECT'] = PROJECT
os.chdir(PIPELINE_DIR)
print('cwd:', os.getcwd())

## 1. Earth Engine pipeline (exports to Google Drive)
Authenticate once, then launch the batch exports. Skip/rerun as needed — this is the only quota-gated stage.

In [ ]:
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print('EE ready:', ee.String('ok').getInfo())

### Stage 1 · Submit the Earth Engine exports

**What this stage does.** Submits the planting, WRSI and zonal exports for the product. This is the
only quota-gated stage. Comment it out once the exports exist.

**Three outputs per product**, landing in Drive under `planting_outputs/`:

* `planting_<country>_<crop>_<season>_<year>` — per-pixel planting dekad, GeoTIFF
* `wrsi_<...>` — WRSI, deficit in mm, and the crop-performance class
* `<...>_zonal` — admin-1 modal, p10, p50 and p90 planting dekad, CSV

**Expected time.** Minutes to hours. The monitor cell below refreshes on demand; more than about 20
minutes with nothing entering RUNNING means a stalled queue rather than a slow one.

In [ ]:
# Launch planting + WRSI + zonal exports for the product(s). Comment out if already exported.
!python run.py --year {YEAR} --country {COUNTRY} --crop {CROP}

In [ ]:
# Monitor export tasks (re-run to refresh)
seen = {}
for op in ee.data.listOperations():
    md = op.get('metadata', {}); d = md.get('description', '')
    if f'{COUNTRY}_{CROP}' in d:
        ct = md.get('createTime', '')
        if d not in seen or ct > seen[d][0]:
            seen[d] = (ct, md.get('state', '?'))
for d in sorted(seen):
    print(f'{seen[d][1]:10} {d}')

## 2. Local maps from the exported GeoTIFFs
Runs entirely locally (rasterio + matplotlib). Downsamples the ~32k×32k tiles on read, so memory stays small.

### Stage 2 · Local maps

**What this stage does.** Renders the exported tiles into a styled PDF, downsampling on read so a
32,000 by 32,000 tile does not exhaust memory, then previews the last page inline.

**Expected output.** `Planting_Maps.pdf` and an inline preview. A preview that is almost entirely
blank means the GeoTIFF is a shard rather than the mosaic; check that the export finished and that all
shards were downloaded.

In [ ]:
!python render_maps_pdf.py --outputs-dir "{OUTPUTS_DIR}" --out Planting_Maps.pdf

from IPython.display import Image, display
# convert the map page to PNG and show inline
import fitz
doc = fitz.open('Planting_Maps.pdf')
doc[len(doc)-1].get_pixmap(dpi=150).save('_preview_map.png'); doc.close()
display(Image('_preview_map.png'))

## 3. Admin-level statistics + skill vs FEWS/FAO calendar
Levels: 1 = county, 2 = constituency, 3 = ward. Each produces a CSV + a 2-page PDF
(modal planting dekad + calendar hit-rate).

### Stage 3 · Administrative statistics and calendar skill

**What this stage does.** Aggregates the planting dekad to three levels — 1 county, 2 constituency,
3 ward — and scores each unit against the calendar window. Each level writes a CSV and a two-page PDF:
the modal planting dekad, and the calendar hit rate.

**Definitions.** Hit rate is the share of maize area inside `CAL_WIN`; bias is the mean signed
difference in dekads, positive being later than the calendar.

**Expected values, Kenya long rains 2024, level 2, 253 constituencies.** Modal dekad **8**, mean **8.05**,
p10 **7.3**, p90 **8.8**, mean hit rate **0.73**, mean bias **−1.95 dekads**, mean absolute error
**2.01 dekads** against the calendar window.

**The negative bias is a property of the calendar, not an error in the estimate.** The indicative
window opens at Mar-d2 while most western farmers plant in Mar-d1 to Mar-d3. Against **farmer-reported**
dates rather than the calendar, the same product carries a bias of only **−0.31 dekads** and an MAE of
**1.02 dekads**, with 93 % of counties within two dekads. When you report accuracy, report the farmer
comparison and say which one you used.

In [ ]:
ws, we = CAL_WIN
for level in (1, 2, 3):
    print(f'\n===== admin level {level} =====')
    !python admin_skill_local.py --level {level} --product {PRODUCT} --win {ws} {we} --outputs-dir "{OUTPUTS_DIR}"

import pandas as pd
pd.read_csv(f'{PRODUCT}_L1_skill.csv').head(10)

In [ ]:
# show the county modal-dekad + hit-rate maps inline
doc = fitz.open(f'{PRODUCT}_L1_skill.pdf')
for i in range(doc.page_count):
    doc[i].get_pixmap(dpi=140).save(f'_preview_L1_p{i}.png')
doc.close()
for i in range(2):
    display(Image(f'_preview_L1_p{i}.png'))

## 4. AEZ → maize maturity class
Overlays planting dekad with Kenya's Jaetzold/Sombroek AEZ. The AEZ code (temperature belt +
moisture zone) sets the length of growing period → indicative early / medium / late-maturing maize.

**Reading the result:** planting *timing* is nearly uniform across zones (long-rains onset); the
*maturity class* is what varies by AEZ. So the maturity-class map (page 2), not the dekad map,
is what depicts early/mid/late varieties.

### Stage 4 · Agro-ecological zone and maturity class

**What this stage does.** Overlays the planting dekad with Kenya's agro-ecological zones. The zone
code, a temperature belt and a moisture zone, sets the length of growing period, which in turn implies
an early, medium or late-maturing maize variety.

**Read page 2, not page 1.** Planting **timing** is nearly uniform across zones, because the long-rains
onset is regional. It is the **maturity class** that varies by zone. The dekad map therefore looks
flat, and that flatness is the finding.

**A caveat worth carrying.** A growing-degree-day clock, tested against this fixed-length assumption,
puts the long-rains highland growing period out by about 68 days. The maturity classes here come from
the zone table, not from a thermal-time model, and should be treated as indicative.

In [ ]:
!python aez_analysis.py --product {PRODUCT} --outputs-dir "{OUTPUTS_DIR}" --px 4000

doc = fitz.open(f'{PRODUCT}_AEZ_maturity.pdf')
for i in range(doc.page_count):
    doc[i].get_pixmap(dpi=140).save(f'_preview_AEZ_p{i}.png')
doc.close()
for i in range(2):
    display(Image(f'_preview_AEZ_p{i}.png'))

In [ ]:
import pandas as pd
aez = pd.read_csv(f'{PRODUCT}_AEZ_maturity.csv')
# pixel-weighted modal planting dekad by maturity class
g = aez.groupby('maturity_class').apply(
    lambda d: pd.Series({'pixels': d.n_px.sum(),
                         'wtd_modal_dekad': round((d.modal_dekad*d.n_px).sum()/d.n_px.sum(), 2),
                         'n_zones': d.aez.nunique()}))
g

## 5. Skill-strength graphs
Per-output skill breakdown (ranked county hit-rate, distributions, bias) and the AEZ influence
on planting timing. The maps must have been generated first (section 3).

### Stage 5 · Skill graphs

**What this stage does.** Breaks the skill down: ranked county hit rate, the ward-level distribution,
the bias, and a scatter of estimated against observed. Stage 3 must have run first.

**What to look for.** A wide spread in county hit rate with no geographic pattern is noise. A block of
low-skill counties that share a season regime is a real finding, and the usual cause is a calendar
window that does not fit that regime; Kenya has three regimes, not one, and 39.4 % of the mapped
short-rains area is in fact the standing long-rains crop. See `KENYA_SEASON_REGIMES.md`.

In [ ]:
# per-output skill detail (Long rains): ranked county hit-rate, ward distribution, bias, scatter
!python skill_graphs.py --product {PRODUCT} --win {ws} {we}
display(Image(f'{PRODUCT}_skill_graphs.png'))

### Stage 5b · How much the zone actually moves planting

Boxplots of planting dekad by maturity class and altitude belt. The expected result is **overlapping
boxes**: zones differ in how long the crop takes, not in when it goes in. A strong separation here
would contradict stage 4 and would be worth investigating before it is reported.

In [ ]:
# AEZ influence on planting TIMING (boxplots by maturity class & altitude belt)
!python aez_influence.py --product {PRODUCT} --outputs-dir "{OUTPUTS_DIR}" --px 4000
display(Image(f'{PRODUCT}_AEZ_influence.png'))

### 5b. Skill strength ACROSS outputs (seasons / products)
Compares every `*_stats.csv` (pixel-based skill, each scored vs its own calendar window) so you
can see which output is stronger. Needs the `stats.py` exports in your Drive `planting_outputs/`.

### Stage 5c · Compare products against each other

**What this stage does.** Scores every `*_stats.csv` in the stats folder, each against its own calendar
window, so seasons and products can be ranked.

**How to read it.** A season with a wide or badly placed calendar window scores low even when the
estimate is good, because the window is the yardstick. Use this to find which **window** needs
revisiting, then confirm with farmer data before changing the estimate.

In [ ]:
STATS_DIR = OUTPUTS_DIR  # or your Drive '.../My Drive/planting_outputs' if stats.csv live there
!python skill_across_outputs.py --stats-dir "{STATS_DIR}" --out skill_across_outputs.png
display(Image('skill_across_outputs.png'))

## 6. To run other seasons / products
- **Kenya Short rains:** `PRODUCT = 'planting_Kenya_maize_Shortrains_2024_250m'`, `CAL_WIN = (28, 32)`.
- **Ethiopia Meher (unimodal):** `PRODUCT = 'planting_Ethiopia_maize_Meher_2024_250m'`, `CAL_WIN = (10, 15)` (Apr-d1–May-d3); use Ethiopia GADM for the admin overlay.
- **250 m products** carry the `_250m` tag (production scale). Re-run stages 2–4 once the GeoTIFF has exported.
- **WRSI maps:** once `wrsi_*` GeoTIFFs are exported, `render_maps_pdf.py` picks them up automatically.
- Everything in stages 2–4 is local and quota-free.
